### Regression task to learn better 
This notebook combines two implementations
- kfold on tf-idf vectorization + ridge model
- kfold on roberta-base 

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error

In [ ]:
DIR = "/kaggle/input/competitions/commonlitreadabilityprize"

train_df = pd.read_csv(f"{DIR}/train.csv")
test_df = pd.read_csv(f"{DIR}/test.csv")

print(train_df.head())

In [ ]:
kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
oof_preds = np.zeros(len(train_df)) # array for predictions init with 0s
test_preds = np.zeros(len(test_df))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
    X_tr, y_tr = train_df.iloc[tr_idx]["excerpt"], train_df.iloc[tr_idx]["target"]
    X_val = train_df.iloc[val_idx]["excerpt"]

    model = make_pipeline(
        TfidfVectorizer(ngram_range = (1, 2), max_features = 20000, sublinear_tf = True),
        Ridge(alpha = 1.0)
    )

    model.fit(X_tr, y_tr)

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test_df["excerpt"]) / 5.0

In [ ]:
cv_rmse_score = root_mean_squared_error(train_df["target"], oof_preds)
print(cv_rmse_score)

In [ ]:
submission = pd.DataFrame({
    "id" : test_df["id"],
    "target" : test_preds
})

submission.to_csv("submission.csv", index = False)
print(submission.head())

In [ ]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 256


In [ ]:
from transformers import(
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df = train_df.rename(columns = {"target" : "label"})
train_df["label"] = train_df["label"].astype("float32")


In [ ]:
from datasets import Dataset

def tokenize_fn(batch):
    return tokenizer(batch["excerpt"], truncation = True, max_length = MAX_LENGTH)

test_ds = Dataset.from_pandas(test_df[["excerpt"]]).map(
    tokenize_fn, 
    batched = True,
    remove_columns = ["excerpt"]
)

In [ ]:
from transformers import DataCollatorWithPadding

data_colator = DataCollatorWithPadding(tokenizer = tokenizer)

In [ ]:
!pip install -q evaluate

In [ ]:
import evaluate

accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = 1)

    rmse = root_mean_squared_error(labels, logits.squeeze())

    accuracy = accuracy_metric.compute(
        predictions = predictions,
        references = labels
    )
    
    return {
        "rmse" : rmse,
        "accuracy" : accuracy["accuracy"]
    }



In [ ]:
# remake the kfold

kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
oof_preds = np.zeros(len(train_df)) # array for predictions init with 0s
test_preds = np.zeros(len(test_df))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):

    train_ds = Dataset.from_pandas(
        train_df.iloc[tr_idx][["excerpt", "label"]].reset_index(drop = True)
    ).map(
        tokenize_fn,
        batched = True,
        remove_columns = ["excerpt"]
    )

    val_ds = Dataset.from_pandas(
        train_df.iloc[val_idx][["excerpt", "label"]].reset_index(drop = True)
    ).map(
        tokenize_fn,
        batched = True,
        remove_columns = ["excerpt"]
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels = 1
    )

    args = TrainingArguments(
        output_dir="/kaggle/working/commonlitreadability_model",

        # base hyperparams
        num_train_epochs=3,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        fp16 = True,
        weight_decay = 0.01,
    
        # evaluation and saving
        eval_strategy = "epoch",
        save_strategy = "epoch",
        save_total_limit = 1, # only save one checkpoint
        load_best_model_at_end = True, # gives us the best checkpoint of the model
        metric_for_best_model = "rmse",
    
        # logging
        logging_steps = 50, # log every 50 steps
        report_to = "none"
    )

    trainer = Trainer(
        model = model,
        args = args,
        train_dataset = train_ds,
        eval_dataset = val_ds,
        data_collator = data_colator,
        compute_metrics = compute_metrics
    )

    trainer.train()

    oof_preds[val_idx] = trainer.predict(val_ds).predictions.squeeze()
    test_preds += trainer.predict(test_ds).predictions.squeeze() / 5.0




In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "target": test_preds
})

submission.to_csv("submission.csv", index = False)
print(submission.head(1))

In [ ]:
# merge the two solutions
oof_blend = 0.90 * oof_preds + 0.10 * oof_preds_ridge
test_blend = 0.90 * test_preds + 0.10 * test_preds_ridge
